# Dynamic Time Warping (DTW)


#### Properties

DTW holds a few of the basic metric properties, such as:

- $\text{DTW}(\boldsymbol{x}, \boldsymbol{y}) \geq 0$

- $\text{DTW}(\boldsymbol{x}, \boldsymbol{x}) = 0$

- $\text{DTW}(\boldsymbol{x}, \boldsymbol{y}) = \text{DTW}(\boldsymbol{y}, \boldsymbol{x})$

However, DTW **does not** satisfy neither the triangular inequality nor the identity of indiscernibles, thus, mathematically speaking, DTW is not a valid distance metric.

**Reference**:

- Romain Tavenard, An introduction to Dynamic Time Warping, [site](https://rtavenar.github.io/blog/dtw.html)

- Dynamic Time Warping, [site](https://rtavenar.github.io/ml4ts_ensai/contents/align/dtw.html)

Python packages:

- `tslearn`, Dynamic Time Warping, [site](https://tslearn.readthedocs.io/en/stable/user_guide/dtw.html)

- `DTAIDistance`, Dynamic Time Warping (DTW), [site](https://dtaidistance.readthedocs.io/en/latest/index.html)

In [1]:
import time
import numpy as np
import pandas as pd

In [2]:
# load data
pattern = pd.read_csv(r'C:\Users\Wei Zhou\Documents\zhouwei file\Github-Project\VeNote-Machine-Learning\clustering\test_data\pattern_processed_moving_sum.csv', index_col=0, header=[0, 1, 2])
print('Data shape:', pattern.shape)
# convert the "t_slot" to int
pattern = pattern.rename(columns = lambda x : int(x), level=2, inplace=False)

Data shape: (1089, 1108)


# 2. `tslearn` vs. `DTAIDistance`

`tslearn.metrics.dtw` can only compute DTW for two time series but not available for large set of time series.

`dtaidistance.dtw.distance_matrix_fast` can effectively compute DTW for a large set of time series.


## 2.1 DTW distance

In [28]:
from clustering.code_utils.distance_metric.combined_euclidean_dtw import compute_dtw_matrix

In [29]:
data = pattern.values.copy()[:500, :277]
print('Data shape:', data.shape)

n_sample = len(data)
# Make sure the data type is np.double
ts1 = [data[i].astype(np.double) for i in range(200)]
ts2 = [data[i].astype(np.double) for i in range(200, 500)]

Data shape: (500, 277)


#### (1) Inner distance matrix

In [30]:
# Compute DTW matrix using `tslearn`
t_start = time.time()
dist_1 = compute_dtw_matrix(ts1, n_jobs=31, backend_package='tslearn')
print('\nTime elapsed:', time.time() - t_start,
    '\nPresent whether DTW is symmetric:', np.allclose(dist_1, dist_1.T))


# Compute DTW matrix using `DTAIDistance`
t_start = time.time()
dist_2 = compute_dtw_matrix(ts1, backend_package='dtai')
print('\nTime elapsed:', time.time() - t_start,
    '\nPresent whether DTW is symmetric:', np.allclose(dist_2, dist_2.T),
    '\nDifference:', np.allclose(dist_1, dist_2))


Time elapsed: 5.5992114543914795 
Present whether DTW is symmetric: True

Time elapsed: 0.4567728042602539 
Present whether DTW is symmetric: True 
Difference: True


#### (2) Distance between two sets of time series

In [31]:
# Compute DTW matrix using `tslearn`
t_start = time.time()
dist_1 = compute_dtw_matrix(ts1, ts2, n_jobs=31, backend_package='tslearn')
print('\nTime elapsed:', time.time() - t_start)

# Compute DTW matrix using `DTAIDistance`
t_start = time.time()
dist_2 = compute_dtw_matrix(ts1, ts2, backend_package='dtai')
print('\nTime elapsed:', time.time() - t_start,
    '\nDifference:', np.allclose(dist_1, dist_2))


Time elapsed: 3.929936408996582

Time elapsed: 0.7068848609924316 
Difference: True


#### (3) Using parameter `window` to reduce the computation complexity

In [41]:
data = pattern.values.copy()[:500]
ts1 = [data[i].astype(np.double) for i in range(len(data))]

t_start = time.time()
dist_1 = compute_dtw_matrix(ts1, backend_package='dtai')
print('\nTime elapsed:', time.time() - t_start)

t_start = time.time()
dist_2 = compute_dtw_matrix(ts1, backend_package='dtai', window=12*4)
print('\nTime elapsed:', time.time() - t_start)

print('\nDifference:', np.sum(np.abs(dist_1, dist_2)))


Time elapsed: 35.22828722000122

Time elapsed: 2.9768781661987305

Difference: 127619023.83760789


## 2.2 DTW Barycenter

In [3]:
from dtaidistance.dtw_barycenter import dba as dtai_dba
from dtaidistance.dtw_barycenter import dba_loop as dtai_loop_dba
from tslearn.barycenters import dtw_barycenter_averaging as tslearn_dba

In [4]:
data = pattern.values.copy()[:500]
ts1 = [data[i].astype(np.double) for i in range(len(data))]

In [ ]:
# Compute the barycenter of a set of time series
t_start = time.time()
centroid_1 = dtai_loop_dba(ts1, c=None, max_it=10, thr=0.001)
print('\nTime elapsed:', time.time() - t_start)

t_start = time.time()
centroid_2 = tslearn_dba(ts1, max_iter=30, tol=1e-05)
print('\nTime elapsed:', time.time() - t_start)

KeyboardInterrupt: 